# Chunking

In [1]:
import torch
import lancedb
from lancedb.embeddings import get_registry
from lancedb.pydantic import LanceModel, Vector
from lancedb.rerankers import ColbertReranker
import ollama
import os
import json
from tqdm.notebook import tqdm
import re, unicodedata


def clean_docling_chunk_strings(chunks):
    cleaned_chunks = []
    
    for chunk in chunks:
        # 2️⃣ Normalize Unicode and replace problematic punctuation
        chunk = unicodedata.normalize("NFKD", chunk).replace("\u00A0", " ")
        chunk = chunk.translate(str.maketrans({
            "–": "-", "—": "-", "‘": "'", "’": "'", "“": '"', "”": '"'
        }))

        # 3️⃣ Remove URLs (massive tokenizers killers)
        chunk = re.sub(r"http\S+", "", chunk)

        # 4️⃣ Normalize whitespace but preserve paragraphs
        chunk = re.sub(r"[ \t]+", " ", chunk)
        chunk = re.sub(r"\n\s*\n", "\n\n", chunk)  # merge single newlines, keep double
        chunk = chunk.strip()

        cleaned_chunks.append(chunk)

    return cleaned_chunks



EMBEDDING_MODEL_NAME = "nomic-ai/nomic-embed-text-v1.5"
OLLAMA_MODEL_NAME= "anthropic_chunking"
CHUNKS_WITH_METADATA_FILE_NAME = "preprocessed_chunks/anthropic_control_chunks_with_metadata.json"
INPUT_DIR = "split_documents"
TABLE_NAME = "anthropic_control_table"

study_names = [f for f in os.listdir(INPUT_DIR) if f.endswith('.json')]
processed_chunks=[]
try:
    with open(CHUNKS_WITH_METADATA_FILE_NAME, "r", encoding="utf-8") as f:
        processed_chunks = json.load(f)
except FileNotFoundError:
    print(f"No existing {CHUNKS_WITH_METADATA_FILE_NAME} file found, starting fresh.")
    

chunks_with_metadata = processed_chunks.copy()
processed_studies = set(chunk["document"] for chunk in processed_chunks)

study_names = [f for f in study_names if f not in processed_studies]
print(f"Found {len(processed_studies)} studies which are already processed.\nStudies which STILL need to be processed: {len(study_names)}:\n{study_names}...")


No existing preprocessed_chunks/anthropic_control_chunks_with_metadata.json file found, starting fresh.
Found 0 studies which are already processed.
Studies which STILL need to be processed: 25:
['Stock_Market_Prediction_via_Multi-Source_Multiple_Instance_Learning.pdf.json', 'A_Conceptual_Framework_and_Recommendations_for_Open_Data_and_Artifacts_in_Empirical_Software_Engineering.pdf.json', 'A_Hybrid_Gaze_Distance_Estimation_via_Cross-Reference_of_Vergence_and_Depth.pdf.json', 'A_Feature_Fusion_Based_Indicator_for_Training-Free_Neural_Architecture_Search.pdf.json', 'A_Resource_Allocation_Model_Based_on_Trust_Evaluation_in_Multi-Cloud_Environments.pdf.json', 'Quantitative_Evaluation_of_Line-Edge_Roughness_in_Various_FinFET_Structures_Bayesian_Neural_Network_With_Automatic_Model_Selection.pdf.json', 'Probabilistic_Artificial_Neural_Network_for_Line-Edge-Roughness-Induced_Random_Variation_in_FinFET.pdf.json', 'Transformation_of_Non-Euclidean_Space_to_Euclidean_Space_for_Efficient_Learning_

# Creating chunks and adding Metadata

As well as semantic context with ollama (Anthropic style)

In [ ]:
for source in tqdm(study_names, desc="Chunking documents..."):   
    with open(f"{INPUT_DIR}/{source}", "r", encoding="utf-8") as f:
        chunks = json.load(f)
    chunks_str = [chunk["text"] for chunk in chunks]
    chunks_str = clean_docling_chunk_strings(chunks_str)
    entire_doc = " ".join(chunks_str)
    print(f"{len(entire_doc)=}\n{entire_doc[:500]=}...")

    for chunk in tqdm(chunks, desc=f"Adding context for chunks of {source[:20]}...", leave=False):    
        chunk_index = chunks.index(chunk)

        entire_doc = "FULL DOCUMENT:\n" + entire_doc
        ollama_prompt = f"CHUNK:\n{chunks_str[chunk_index]}"
        history =  [{'role': 'user', 'content': entire_doc}, {'role': 'user', 'content': ollama_prompt}]

        response = ollama.chat(
            model=OLLAMA_MODEL_NAME,
            messages=history,
            options={
                "num_ctx": 30_000
            }
        )
        context = response['message']['content']
        text_to_embed = context + "\n\n" + chunks_str[chunk_index] 

        chunks_with_metadata.append({'text': text_to_embed, 'original_text':chunks_str[chunk_index], 'context':context, 'document':chunk['document'], 'id': chunk['id']})
        
# Total runtime: 71m 34s for 25 documents

In [4]:
# Save the the processed chunks in case VectorDB upload goes wrong.
# Luckily since this is a notebook, if the chunking is interrupted, we can still save the partial results here.
# Append new chunks to the existing file if it exists, otherwise create it
if os.path.exists(CHUNKS_WITH_METADATA_FILE_NAME):
    print(f"Appending to existing {CHUNKS_WITH_METADATA_FILE_NAME} file.")
    with open(CHUNKS_WITH_METADATA_FILE_NAME, "r", encoding="utf-8") as f:
        existing_data = json.load(f)
    # Avoid duplicate entries by id
    existing_ids = {chunk['id'] for chunk in existing_data}
    new_chunks = [chunk for chunk in chunks_with_metadata if chunk['id'] not in existing_ids]
    chunks_with_metadata = existing_data + new_chunks

with open(CHUNKS_WITH_METADATA_FILE_NAME, "w", encoding="utf-8") as f:
    json.dump(chunks_with_metadata, f, ensure_ascii=False, indent=2)

print(f"Results saved to {CHUNKS_WITH_METADATA_FILE_NAME}")

Results saved to preprocessed_chunks/anthropic_control_chunks_with_metadata.json


# Creating Database

In [5]:
registry = get_registry()
hf = registry.get("huggingface").create(name=EMBEDDING_MODEL_NAME, trust_remote_code=True, device="cuda" if torch.cuda.is_available() else "cpu")


# Define model
class MyDocument(LanceModel):
    text: str = hf.SourceField()
    vector: Vector(hf.ndims()) = hf.VectorField()
    original_text: str
    context: str
    document: str
    id: str  # Unique identifier for the chunk


db = lancedb.connect("./db")
db.create_table(TABLE_NAME, schema=MyDocument, mode="overwrite") # Uncomment this line when running this cell for the first time
table = db.open_table(TABLE_NAME)

# Upload in batches with progress bar
with open(CHUNKS_WITH_METADATA_FILE_NAME, "r", encoding="utf-8") as f:
    chunks_with_metadata = json.load(f)

batch_size = 100
for i in tqdm(range(0, len(chunks_with_metadata), batch_size), desc="Uploading chunks to VectorDB"):
    batch = chunks_with_metadata[i:i+batch_size]
    table.add(batch)

table.create_scalar_index("id", replace=True) # Index based on the chunk's id, used to manually prevent duplicates

reranker = ColbertReranker()
table.create_fts_index("text", replace=True) # Used by the reranker as well as the hybrid search's BM25 index
table.wait_for_index(["text_idx"])  # Wait for the indexing to finish

<All keys matched successfully>
[2026-01-18T16:28:27Z WARN  lance::dataset::write::insert] No existing dataset at /home/martin/projects/Quantwise/Quantwise-Chunking/db/anthropic_control_table.lance, it will be created


Uploading chunks to VectorDB:   0%|          | 0/4 [00:00<?, ?it/s]

<All keys matched successfully>
<All keys matched successfully>
<All keys matched successfully>
<All keys matched successfully>


Loading ColBERTRanker model colbert-ir/colbertv2.0 (this message can be suppressed by setting verbose=0)
No device set
Using device cuda
No dtype set
Using dtype torch.float32
Loading model colbert-ir/colbertv2.0, this might take a while...
Linear Dim set to: 128 for downcasting


# Example query

In [6]:
prompt = "How was stock market data gathered?"
results = table.search(prompt, query_type="hybrid", vector_column_name="vector", fts_columns="text") \
            .rerank(reranker=reranker) \
            .limit(5) \
            .to_pandas()


results

<All keys matched successfully>


,text,vector,original_text,context,document,id,_relevance_score
0,Details the data collection process for the st...,"[0.70299774, 1.1682792, -3.7868931, -0.2385918...",We collected stock market-related information ...,Details the data collection process for the st...,Stock_Market_Prediction_via_Multi-Source_Multi...,4cf733a743ce1b6eb4e3c41e23b999ed51cd3d280449ef...,1.036964
1,Introduces the central thesis about using Goog...,"[0.40788847, 1.8804536, -3.7192028, -0.2992431...","SUBJECT AREAS:\nSTATISTICAL PHYSICS, THERMODYN...",Introduces the central thesis about using Goog...,srep01684.pdf,326e42cc95fc78ae06dc4023c715a91f02054804d2d494...,0.989308
2,Quantifies the relationship between search vol...,"[0.46575984, 2.3796456, -3.5407345, 0.02398393...","In summary, our results are consistent with th...",Quantifies the relationship between search vol...,srep01684.pdf,c80c4fb4f9449b543440f05257d57a5bee14a10d6f666e...,0.961116
3,"This section details the experimental design, ...","[0.6002093, 1.2585888, -2.9446952, -0.8818481,...",Experimental design. Our paper relates to rese...,"This section details the experimental design, ...",s41598-020-77823-3.pdf,4eecb9240c936f76259c30feaf4292800c84483b696ec2...,0.927112
4,Introduces the core methodology: analyzing Goo...,"[0.66251045, 1.8534836, -3.097365, -0.6566481,...",We analyze the performance of a set of 98 sear...,Introduces the core methodology: analyzing Goo...,srep01684.pdf,31c6e564a2d23c8266ccfc78cef7934798ef7025766d2c...,0.866546


In [ ]:
results.iloc[0,0]

In [ ]:
table.stats()